# Google Data Engineer Interview — Tomorrow Master Notebook

This is a focused interview notebook built from the five uploaded Google/DataInterview question lists.

The PDFs mainly provide **titles, topics, and difficulty**, not full locked statements.  
So this notebook uses the standard interview version of each named problem and prioritizes the questions most useful for a Google Data Engineer interview:

- streaming and state
- event time and watermarks
- SQL patterns
- hash maps
- heaps
- sliding windows
- intervals
- graphs
- dynamic programming
- binary search on answer

Every section contains:

1. pattern recognition
2. what to say aloud
3. best practical solution
4. runnable code
5. tests
6. time and space complexity

## Interview speaking formula

> “I’ll clarify the input and constraints first.  
> The simple solution is ____. Its bottleneck is ____.  
> I’ll optimize using ____.  
> The invariant/state is ____.  
> Time is ____ and space is ____.  
> I’ll test normal, edge, and failure cases.”

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict, deque, OrderedDict
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple
import bisect
import hashlib
import heapq
import math
import random
import sqlite3
import statistics
import threading
import time

random.seed(42)
print("Setup complete")

---
## 1. Two Sum

### How to recognize it
Unsorted array + target + return indices → hash map of seen value to index.

### What to say in the interview
> I’ll scan once. For each value, I check whether its complement was seen earlier. This avoids the O(n²) nested loop.

### Complexity
O(n) average time, O(n) space.

In [ ]:
def two_sum(nums: List[int], target: int) -> List[int]:
    seen = {}
    for i, value in enumerate(nums):
        need = target - value
        if need in seen:
            return [seen[need], i]
        seen[value] = i
    return []

assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 3], 6) == [0, 1]
print("Two Sum passed")

---
## 2. Two Sum II — Sorted Input

### How to recognize it
Sorted array + pair sum → two pointers.

### What to say in the interview
> If the sum is too small, only moving left rightward can increase it. If too large, only moving right leftward can decrease it.

### Complexity
O(n) time, O(1) space.

In [ ]:
def two_sum_sorted(nums: List[int], target: int) -> List[int]:
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return [left + 1, right + 1]
        if total < target:
            left += 1
        else:
            right -= 1
    return []

assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
print("Two Sum II passed")

---
## 3. Binary Search

### How to recognize it
Sorted array + exact target → binary search.

### What to say in the interview
> I maintain the invariant that the target, if present, remains inside [left, right].

### Complexity
O(log n) time, O(1) space.

In [ ]:
def binary_search(nums: List[int], target: int) -> int:
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

assert binary_search([-1, 0, 3, 5, 9, 12], 9) == 4
assert binary_search([1, 2, 3], 4) == -1
print("Binary Search passed")

---
## 4. Subarray Sum Equals K

### How to recognize it
Count contiguous subarrays with exact sum; negatives allowed → prefix sum frequency map.

### What to say in the interview
> If current prefix is P, every earlier prefix P-k creates one valid subarray ending here.

### Complexity
O(n) time, O(n) space.

In [ ]:
def subarray_sum(nums: List[int], k: int) -> int:
    prefix = 0
    count = 0
    frequencies = {0: 1}
    for value in nums:
        prefix += value
        count += frequencies.get(prefix - k, 0)
        frequencies[prefix] = frequencies.get(prefix, 0) + 1
    return count

assert subarray_sum([1, 1, 1], 2) == 2
assert subarray_sum([1, -1, 0], 0) == 3
print("Subarray Sum passed")

---
## 5. Sliding Window Maximum

### How to recognize it
Maximum in every fixed-size window → monotonic deque of indices.

### What to say in the interview
> I remove expired indices from the front and smaller useless candidates from the back. The front is always the window maximum.

### Complexity
O(n) time, O(k) space.

In [ ]:
def sliding_window_max(nums: List[int], k: int) -> List[int]:
    q = deque()
    result = []
    for i, value in enumerate(nums):
        while q and q[0] <= i - k:
            q.popleft()
        while q and nums[q[-1]] <= value:
            q.pop()
        q.append(i)
        if i >= k - 1:
            result.append(nums[q[0]])
    return result

assert sliding_window_max([1,3,-1,-3,5,3,6,7], 3) == [3,3,5,5,6,7]
print("Sliding Window Maximum passed")

---
## 6. Longest Repeating Character Replacement

### How to recognize it
Longest substring after at most k replacements → variable sliding window.

### What to say in the interview
> A window is valid when window length minus its most frequent character count is at most k.

### Complexity
O(n) time, O(alphabet) space.

In [ ]:
def character_replacement(s: str, k: int) -> int:
    counts = Counter()
    left = 0
    max_frequency = 0
    best = 0

    for right, ch in enumerate(s):
        counts[ch] += 1
        max_frequency = max(max_frequency, counts[ch])

        while (right - left + 1) - max_frequency > k:
            counts[s[left]] -= 1
            left += 1

        best = max(best, right - left + 1)

    return best

assert character_replacement("AABABBA", 1) == 4
print("Character Replacement passed")

---
## 7. Max Consecutive Ones III

### How to recognize it
Longest binary window after flipping at most k zeroes → sliding window counting zeroes.

### What to say in the interview
> I expand right and contract left only when zero count exceeds k.

### Complexity
O(n) time, O(1) space.

In [ ]:
def longest_ones(nums: List[int], k: int) -> int:
    left = 0
    zeroes = 0
    best = 0
    for right, value in enumerate(nums):
        if value == 0:
            zeroes += 1
        while zeroes > k:
            if nums[left] == 0:
                zeroes -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

assert longest_ones([1,1,1,0,0,0,1,1,1,1,0], 2) == 6
print("Max Consecutive Ones III passed")

---
## 8. Decode String

### How to recognize it
Nested encoded string like 3[a2[c]] → stack or recursion.

### What to say in the interview
> I push the current string and repeat count when I see '[', then combine when I see ']'.

### Complexity
O(output size) time, O(n) stack space.

In [ ]:
def decode_string(s: str) -> str:
    count_stack = []
    string_stack = []
    current = ""
    number = 0

    for ch in s:
        if ch.isdigit():
            number = number * 10 + int(ch)
        elif ch == "[":
            count_stack.append(number)
            string_stack.append(current)
            current = ""
            number = 0
        elif ch == "]":
            current = string_stack.pop() + current * count_stack.pop()
        else:
            current += ch

    return current

assert decode_string("3[a2[c]]") == "accaccacc"
print("Decode String passed")

---
## 9. Insert Interval

### How to recognize it
Sorted non-overlapping intervals + one new interval → three phases: before, overlap, after.

### What to say in the interview
> I append intervals before the new one, merge all overlaps into one interval, then append the remainder.

### Complexity
O(n) time, O(n) output space.

In [ ]:
def insert_interval(intervals: List[List[int]], new_interval: List[int]) -> List[List[int]]:
    result = []
    i = 0

    while i < len(intervals) and intervals[i][1] < new_interval[0]:
        result.append(intervals[i])
        i += 1

    while i < len(intervals) and intervals[i][0] <= new_interval[1]:
        new_interval[0] = min(new_interval[0], intervals[i][0])
        new_interval[1] = max(new_interval[1], intervals[i][1])
        i += 1

    result.append(new_interval)
    result.extend(intervals[i:])
    return result

assert insert_interval([[1,3],[6,9]], [2,5]) == [[1,5],[6,9]]
print("Insert Interval passed")

---
## 10. Merge Intervals

### How to recognize it
Merge overlapping ranges → sort by start and compare with last merged interval.

### What to say in the interview
> After sorting, only the last merged interval can overlap the next interval.

### Complexity
O(n log n) time, O(n) output space.

In [ ]:
def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    intervals = sorted(intervals)
    merged = []

    for start, end in intervals:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    return merged

assert merge_intervals([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
print("Merge Intervals passed")

---
## 11. Meeting Rooms

### How to recognize it
Can one person attend all intervals? → sort by start and check overlap with previous end.

### What to say in the interview
> After sorting, any conflict must occur between adjacent meetings.

### Complexity
O(n log n) time, O(1) extra space after sorting.

In [ ]:
def can_attend_meetings(intervals: List[List[int]]) -> bool:
    intervals = sorted(intervals)
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i-1][1]:
            return False
    return True

assert not can_attend_meetings([[0,30],[5,10],[15,20]])
assert can_attend_meetings([[7,10],[2,4]])
print("Meeting Rooms passed")

---
## 12. Course Schedule II

### How to recognize it
Tasks with prerequisites → topological sort using indegrees.

### What to say in the interview
> I repeatedly process zero-indegree courses. If I process all courses, the graph is acyclic.

### Complexity
O(V+E) time, O(V+E) space.

In [ ]:
def find_order(num_courses: int, prerequisites: List[List[int]]) -> List[int]:
    graph = defaultdict(list)
    indegree = [0] * num_courses

    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    q = deque(i for i in range(num_courses) if indegree[i] == 0)
    order = []

    while q:
        node = q.popleft()
        order.append(node)
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                q.append(neighbor)

    return order if len(order) == num_courses else []

assert find_order(2, [[1,0]]) == [0,1]
assert find_order(2, [[1,0],[0,1]]) == []
print("Course Schedule II passed")

---
## 13. Graph Valid Tree

### How to recognize it
Undirected graph is a tree iff it has n-1 edges and is fully connected.

### What to say in the interview
> I first reject the wrong edge count, then run one traversal to verify all nodes are reachable.

### Complexity
O(V+E) time, O(V+E) space.

In [ ]:
def valid_tree(n: int, edges: List[List[int]]) -> bool:
    if len(edges) != n - 1:
        return False

    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)

    seen = {0}
    stack = [0]

    while stack:
        node = stack.pop()
        for neighbor in graph[node]:
            if neighbor not in seen:
                seen.add(neighbor)
                stack.append(neighbor)

    return len(seen) == n

assert valid_tree(5, [[0,1],[0,2],[0,3],[1,4]])
assert not valid_tree(5, [[0,1],[1,2],[2,3],[1,3],[1,4]])
print("Graph Valid Tree passed")

---
## 14. Number of Provinces

### How to recognize it
Count connected components in an adjacency matrix → DFS/BFS.

### What to say in the interview
> Every unvisited city starts one new province; DFS marks the whole component.

### Complexity
O(n²) time, O(n) space.

In [ ]:
def find_circle_num(is_connected: List[List[int]]) -> int:
    n = len(is_connected)
    seen = set()
    provinces = 0

    for city in range(n):
        if city in seen:
            continue
        provinces += 1
        stack = [city]
        seen.add(city)

        while stack:
            current = stack.pop()
            for neighbor, connected in enumerate(is_connected[current]):
                if connected and neighbor not in seen:
                    seen.add(neighbor)
                    stack.append(neighbor)

    return provinces

assert find_circle_num([[1,1,0],[1,1,0],[0,0,1]]) == 2
print("Number of Provinces passed")

---
## 15. 01 Matrix

### How to recognize it
Distance to nearest zero for every cell → multi-source BFS from all zeroes.

### What to say in the interview
> Starting BFS from every zero simultaneously guarantees the first visit gives shortest distance.

### Complexity
O(mn) time, O(mn) space.

In [ ]:
def update_matrix(mat: List[List[int]]) -> List[List[int]]:
    rows, cols = len(mat), len(mat[0])
    distance = [[-1] * cols for _ in range(rows)]
    q = deque()

    for r in range(rows):
        for c in range(cols):
            if mat[r][c] == 0:
                distance[r][c] = 0
                q.append((r, c))

    while q:
        r, c = q.popleft()
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and distance[nr][nc] == -1:
                distance[nr][nc] = distance[r][c] + 1
                q.append((nr, nc))

    return distance

assert update_matrix([[0,0,0],[0,1,0],[1,1,1]]) == [[0,0,0],[0,1,0],[1,2,1]]
print("01 Matrix passed")

---
## 16. Flood Fill

### How to recognize it
Recolor one connected component → DFS/BFS on four neighbors.

### What to say in the interview
> I avoid a separate visited set by recoloring when a cell is visited.

### Complexity
O(mn) time, O(mn) worst-case recursion/queue.

In [ ]:
def flood_fill(image: List[List[int]], sr: int, sc: int, color: int) -> List[List[int]]:
    old = image[sr][sc]
    if old == color:
        return image

    rows, cols = len(image), len(image[0])
    stack = [(sr, sc)]

    while stack:
        r, c = stack.pop()
        if not (0 <= r < rows and 0 <= c < cols):
            continue
        if image[r][c] != old:
            continue

        image[r][c] = color
        stack.extend([(r+1,c),(r-1,c),(r,c+1),(r,c-1)])

    return image

assert flood_fill([[1,1,1],[1,1,0],[1,0,1]], 1, 1, 2) == [[2,2,2],[2,2,0],[2,0,1]]
print("Flood Fill passed")

---
## 17. Koko Eating Bananas

### How to recognize it
Find minimum feasible speed → binary search on answer.

### What to say in the interview
> The required hours decrease monotonically as speed increases, so feasibility is monotonic.

### Complexity
O(n log maxPile) time, O(1) space.

In [ ]:
def min_eating_speed(piles: List[int], h: int) -> int:
    def hours(speed: int) -> int:
        return sum((pile + speed - 1) // speed for pile in piles)

    left, right = 1, max(piles)

    while left < right:
        mid = (left + right) // 2
        if hours(mid) <= h:
            right = mid
        else:
            left = mid + 1

    return left

assert min_eating_speed([3,6,7,11], 8) == 4
print("Koko passed")

---
## 18. Capacity to Ship Packages Within D Days

### How to recognize it
Minimum capacity with monotonic feasibility → binary search on answer.

### What to say in the interview
> Capacity ranges from max package to total sum. If a capacity works, all larger capacities work.

### Complexity
O(n log sum(weights)) time, O(1) space.

In [ ]:
def ship_within_days(weights: List[int], days: int) -> int:
    def required_days(capacity: int) -> int:
        used = 1
        current = 0
        for weight in weights:
            if current + weight > capacity:
                used += 1
                current = 0
            current += weight
        return used

    left, right = max(weights), sum(weights)

    while left < right:
        mid = (left + right) // 2
        if required_days(mid) <= days:
            right = mid
        else:
            left = mid + 1

    return left

assert ship_within_days([1,2,3,4,5,6,7,8,9,10], 5) == 15
print("Shipping Capacity passed")

---
## 19. Coin Change

### How to recognize it
Minimum coins to make amount → 1D dynamic programming.

### What to say in the interview
> dp[x] stores the minimum coins needed for amount x. I build from smaller amounts to larger amounts.

### Complexity
O(amount × coins) time, O(amount) space.

In [ ]:
def coin_change(coins: List[int], amount: int) -> int:
    dp = [amount + 1] * (amount + 1)
    dp[0] = 0

    for total in range(1, amount + 1):
        for coin in coins:
            if coin <= total:
                dp[total] = min(dp[total], dp[total - coin] + 1)

    return -1 if dp[amount] > amount else dp[amount]

assert coin_change([1,2,5], 11) == 3
assert coin_change([2], 3) == -1
print("Coin Change passed")

---
## 20. House Robber

### How to recognize it
Maximum sum with no adjacent selections → rolling DP.

### What to say in the interview
> At each house I choose max(skip current, take current plus best two positions back).

### Complexity
O(n) time, O(1) space.

In [ ]:
def rob(nums: List[int]) -> int:
    two_back = one_back = 0
    for value in nums:
        two_back, one_back = one_back, max(one_back, two_back + value)
    return one_back

assert rob([2,7,9,3,1]) == 12
print("House Robber passed")

---
## 21. Longest Increasing Subsequence

### How to recognize it
Longest increasing subsequence length → patience sorting with binary search.

### What to say in the interview
> `tails[i]` is the smallest possible tail for an increasing subsequence of length i+1.

### Complexity
O(n log n) time, O(n) space.

In [ ]:
def length_of_lis(nums: List[int]) -> int:
    tails = []
    for value in nums:
        i = bisect.bisect_left(tails, value)
        if i == len(tails):
            tails.append(value)
        else:
            tails[i] = value
    return len(tails)

assert length_of_lis([10,9,2,5,3,7,101,18]) == 4
print("LIS passed")

---
## 22. Reconstruct Itinerary

### How to recognize it
Use every directed ticket once and choose lexical order → Eulerian path / Hierholzer.

### What to say in the interview
> I consume edges during DFS and append airports after exhausting outgoing edges, then reverse the route.

### Complexity
O(E log E) time, O(E) space.

In [ ]:
def find_itinerary(tickets: List[List[str]]) -> List[str]:
    graph = defaultdict(list)
    for source, destination in sorted(tickets, reverse=True):
        graph[source].append(destination)

    route = []

    def visit(airport: str):
        while graph[airport]:
            visit(graph[airport].pop())
        route.append(airport)

    visit("JFK")
    return route[::-1]

tickets = [["MUC","LHR"],["JFK","MUC"],["SFO","SJC"],["LHR","SFO"]]
assert find_itinerary(tickets) == ["JFK","MUC","LHR","SFO","SJC"]
print("Itinerary passed")

---
## 23. Path with Maximum Probability

### How to recognize it
Maximum product path in weighted graph → Dijkstra with a max heap.

### What to say in the interview
> I store the best probability found for each node and always expand the currently highest probability path.

### Complexity
O((V+E) log V) time, O(V+E) space.

In [ ]:
def max_probability(
    n: int,
    edges: List[List[int]],
    probabilities: List[float],
    start: int,
    end: int,
) -> float:
    graph = defaultdict(list)

    for (a, b), probability in zip(edges, probabilities):
        graph[a].append((b, probability))
        graph[b].append((a, probability))

    best = [0.0] * n
    best[start] = 1.0
    heap = [(-1.0, start)]

    while heap:
        negative_probability, node = heapq.heappop(heap)
        probability = -negative_probability

        if node == end:
            return probability

        if probability < best[node]:
            continue

        for neighbor, edge_probability in graph[node]:
            candidate = probability * edge_probability
            if candidate > best[neighbor]:
                best[neighbor] = candidate
                heapq.heappush(heap, (-candidate, neighbor))

    return 0.0

result = max_probability(
    3,
    [[0,1],[1,2],[0,2]],
    [0.5,0.5,0.2],
    0,
    2,
)
assert abs(result - 0.25) < 1e-9
print("Maximum Probability passed")

---
## 24. Watermark Progress Tracker

### How to recognize it
Streaming event-time progress → track max event time per partition and subtract allowed lateness.

### What to say in the interview
> The safe global watermark is the minimum partition watermark so slow partitions are not skipped.

### Complexity
O(1) update average; O(P) state for P partitions.

In [ ]:
class PartitionWatermark:
    def __init__(self, allowed_lateness: int):
        self.allowed_lateness = allowed_lateness
        self.max_event_time = {}

    def update(self, partition: str, event_time: int) -> int:
        self.max_event_time[partition] = max(
            event_time,
            self.max_event_time.get(partition, -math.inf),
        )
        return self.global_watermark()

    def global_watermark(self) -> int:
        if not self.max_event_time:
            return -math.inf
        return min(
            value - self.allowed_lateness
            for value in self.max_event_time.values()
        )

tracker = PartitionWatermark(5)
assert tracker.update("p0", 100) == 95
assert tracker.update("p1", 95) == 90
assert tracker.update("p0", 110) == 90
assert tracker.update("p1", 108) == 103
print("Watermark passed")

---
## 25. Late Event Handler

### How to recognize it
Event older than watermark → explicit policy: drop, side output, or update.

### What to say in the interview
> I will not silently discard late data. I’ll expose a policy and route late events to a side output unless retractions are supported.

### Complexity
O(1) per event; O(L) side-output storage.

In [ ]:
@dataclass(order=True)
class Event:
    event_time: int
    processing_time: int
    event_id: str
    payload: Any = None

class LateEventHandler:
    def __init__(self, allowed_lateness: int, policy: str = "side_output"):
        self.allowed_lateness = allowed_lateness
        self.policy = policy
        self.max_event_time = -math.inf
        self.accepted = []
        self.late = []

    @property
    def watermark(self):
        return self.max_event_time - self.allowed_lateness

    def process(self, event: Event) -> str:
        is_late = event.event_time < self.watermark
        self.max_event_time = max(self.max_event_time, event.event_time)

        if not is_late:
            self.accepted.append(event)
            return "accepted"

        if self.policy == "drop":
            return "dropped"

        self.late.append(event)
        return "late_side_output"

handler = LateEventHandler(5)
assert handler.process(Event(100,101,"a")) == "accepted"
assert handler.process(Event(110,111,"b")) == "accepted"
assert handler.process(Event(101,112,"late")) == "late_side_output"
print("Late Event Handler passed")

---
## 26. Sliding Window Distinct Count

### How to recognize it
Distinct count for every fixed window → frequency map for entering/leaving values.

### What to say in the interview
> I update only two elements per slide instead of rebuilding a set.

### Complexity
O(n) time, O(k) space.

In [ ]:
def distinct_counts(values: List[int], k: int) -> List[int]:
    if k <= 0 or k > len(values):
        return []

    counts = Counter(values[:k])
    result = [len(counts)]

    for right in range(k, len(values)):
        outgoing = values[right-k]
        counts[outgoing] -= 1
        if counts[outgoing] == 0:
            del counts[outgoing]

        counts[values[right]] += 1
        result.append(len(counts))

    return result

assert distinct_counts([1,2,1,3,4,2,3], 4) == [3,4,4,3]
print("Distinct Window passed")

---
## 27. Streaming Median Finder

### How to recognize it
Median from continuous stream → two heaps.

### What to say in the interview
> I keep lower half in a max heap and upper half in a min heap, balanced within one element.

### Complexity
Insert O(log n), median O(1), space O(n).

In [ ]:
class MedianFinder:
    def __init__(self):
        self.lower = []
        self.upper = []

    def add(self, value: float):
        if not self.lower or value <= -self.lower[0]:
            heapq.heappush(self.lower, -value)
        else:
            heapq.heappush(self.upper, value)

        if len(self.lower) > len(self.upper) + 1:
            heapq.heappush(self.upper, -heapq.heappop(self.lower))
        elif len(self.upper) > len(self.lower):
            heapq.heappush(self.lower, -heapq.heappop(self.upper))

    def median(self) -> float:
        if len(self.lower) > len(self.upper):
            return float(-self.lower[0])
        return (-self.lower[0] + self.upper[0]) / 2

finder = MedianFinder()
results = []
for value in [5,2,10,4]:
    finder.add(value)
    results.append(finder.median())

assert results == [5.0,3.5,5.0,4.5]
print("Streaming Median passed")

---
## 28. Reservoir Sampling

### How to recognize it
Uniform sample of k items from unknown-size stream → reservoir sampling.

### What to say in the interview
> At stream position i, I replace a random reservoir element with probability k/i, which keeps every item equally likely.

### Complexity
O(n) time, O(k) space.

In [ ]:
def reservoir_sample(stream: Iterable[Any], k: int, seed: int = 42) -> List[Any]:
    rng = random.Random(seed)
    reservoir = []

    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            j = rng.randint(0, i)
            if j < k:
                reservoir[j] = item

    if len(reservoir) < k:
        raise ValueError("stream shorter than k")

    return reservoir

sample = reservoir_sample(range(100), 5)
assert len(sample) == 5
print(sample)

---
## 29. Top K Stream Elements

### How to recognize it
Need only largest k from a stream → min heap of size k.

### What to say in the interview
> The heap root is the smallest element currently in the top k, so it is the eviction candidate.

### Complexity
O(log k) per event, O(k) space.

In [ ]:
class TopKStream:
    def __init__(self, k: int):
        self.k = k
        self.heap = []

    def add(self, value: int):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, value)
        elif value > self.heap[0]:
            heapq.heapreplace(self.heap, value)

    def values(self) -> List[int]:
        return sorted(self.heap, reverse=True)

stream = TopKStream(3)
for value in [5,1,10,3,8,12]:
    stream.add(value)

assert stream.values() == [12,10,8]
print("Top K Stream passed")

---
## 30. Sessionization Engine

### How to recognize it
Group user events into sessions using inactivity gap → sort by user/time and compare with previous event.

### What to say in the interview
> A new session starts when the gap is greater than the threshold.

### Complexity
Batch O(n log n); ordered stream O(n) with O(U) state.

In [ ]:
def sessionize(
    events: List[Tuple[str, int]],
    inactivity_gap: int,
) -> List[Tuple[str, int, int]]:
    events = sorted(events, key=lambda x: (x[0], x[1]))
    last = {}
    session_id = defaultdict(int)
    result = []

    for user, event_time in events:
        if user not in last or event_time - last[user] > inactivity_gap:
            session_id[user] += 1

        result.append((user, event_time, session_id[user]))
        last[user] = event_time

    return result

events = [("u1",0),("u1",10),("u1",50),("u2",3)]
assert sessionize(events, 30) == [
    ("u1",0,1),
    ("u1",10,1),
    ("u1",50,2),
    ("u2",3,1),
]
print("Sessionization passed")

---
## 31. LRU Cache

### How to recognize it
O(1) get/put with eviction → hash map + doubly linked recency order.

### What to say in the interview
> I’ll use OrderedDict semantics: move accessed keys to the end and evict from the front.

### Complexity
O(1) average get/put, O(capacity) space.

In [ ]:
class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key: Any) -> Any:
        if key not in self.data:
            return -1
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key: Any, value: Any):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)

cache = LRUCache(2)
cache.put(1,1)
cache.put(2,2)
assert cache.get(1) == 1
cache.put(3,3)
assert cache.get(2) == -1
print("LRU passed")

---
## 32. Backpressure Flow Controller

### How to recognize it
Producer faster than consumer → bounded queue and explicit overload policy.

### What to say in the interview
> I will bound memory and make overload behavior explicit. Rejecting protects memory; blocking preserves data but increases latency.

### Complexity
O(1) offer/poll, O(capacity) space.

In [ ]:
class BackpressureController:
    def __init__(self, capacity: int, policy: str = "reject"):
        self.capacity = capacity
        self.policy = policy
        self.queue = deque()

    def offer(self, event: Any) -> bool:
        if len(self.queue) < self.capacity:
            self.queue.append(event)
            return True

        if self.policy == "reject":
            return False

        if self.policy == "drop_oldest":
            self.queue.popleft()
            self.queue.append(event)
            return True

        raise ValueError("unsupported policy")

    def poll(self) -> Optional[Any]:
        return self.queue.popleft() if self.queue else None

controller = BackpressureController(2, "drop_oldest")
controller.offer("a")
controller.offer("b")
controller.offer("c")
assert list(controller.queue) == ["b","c"]
print("Backpressure passed")

---
## 33. Bloom Filter

### How to recognize it
Approximate membership with bounded memory → bit array + multiple hashes.

### What to say in the interview
> A zero bit proves absence. All one bits mean probably present. False positives are possible, false negatives are not.

### Complexity
O(h) add/query, O(m) bits.

In [ ]:
class BloomFilter:
    def __init__(self, bit_count: int = 10000, hash_count: int = 7):
        self.bit_count = bit_count
        self.hash_count = hash_count
        self.bits = bytearray((bit_count + 7) // 8)

    def _indexes(self, item: Any):
        raw = repr(item).encode()
        h1 = int.from_bytes(hashlib.md5(raw).digest(), "big")
        h2 = int.from_bytes(hashlib.sha1(raw).digest(), "big")
        for i in range(self.hash_count):
            yield (h1 + i * h2 + i * i) % self.bit_count

    def add(self, item: Any):
        for index in self._indexes(item):
            self.bits[index // 8] |= 1 << (index % 8)

    def might_contain(self, item: Any) -> bool:
        return all(
            self.bits[index // 8] & (1 << (index % 8))
            for index in self._indexes(item)
        )

bf = BloomFilter()
for item in ["apple","banana","mango"]:
    bf.add(item)

assert bf.might_contain("apple")
print("Bloom Filter passed")

---
# 34. SQL Interview Patterns — Runnable SQLite

## What to say

> “Before writing SQL, I define the output grain, join cardinality, and whether ranking occurs before or after aggregation.”

This section covers:

- deduplication
- latest row per key
- anti-join
- conditional aggregation
- top-N per group

In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript('''
CREATE TABLE events (
    user_id TEXT,
    event_time INTEGER,
    event_type TEXT,
    value INTEGER,
    ingestion_time INTEGER
);

INSERT INTO events VALUES
('u1', 10, 'click', 1, 100),
('u1', 10, 'click', 1, 101),
('u1', 50, 'purchase', 20, 102),
('u2', 5, 'click', 1, 103),
('u2', 20, 'click', 1, 104),
('u3', 7, 'view', 1, 105);

CREATE TABLE users (user_id TEXT PRIMARY KEY);
INSERT INTO users VALUES ('u1'), ('u2'), ('u3'), ('u4');

CREATE TABLE sales (
    category TEXT,
    product TEXT,
    quantity INTEGER
);
INSERT INTO sales VALUES
('A','p1',5), ('A','p1',4), ('A','p2',7), ('A','p3',2),
('B','q1',9), ('B','q2',3), ('B','q3',8);
''')

dedup_sql = '''
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY user_id, event_time, event_type
               ORDER BY ingestion_time DESC
           ) AS rn
    FROM events
)
SELECT user_id, event_time, event_type, value, ingestion_time
FROM ranked
WHERE rn = 1
ORDER BY user_id, event_time;
'''

latest_sql = '''
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY event_time DESC, ingestion_time DESC
           ) AS rn
    FROM events
)
SELECT user_id, event_time, event_type
FROM ranked
WHERE rn = 1
ORDER BY user_id;
'''

anti_join_sql = '''
SELECT u.user_id
FROM users u
WHERE NOT EXISTS (
    SELECT 1
    FROM events e
    WHERE e.user_id = u.user_id
)
ORDER BY u.user_id;
'''

pivot_sql = '''
SELECT user_id,
       SUM(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) AS clicks,
       SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases
FROM events
GROUP BY user_id
ORDER BY user_id;
'''

top_n_sql = '''
WITH totals AS (
    SELECT category, product, SUM(quantity) AS total_quantity
    FROM sales
    GROUP BY category, product
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY category
               ORDER BY total_quantity DESC, product ASC
           ) AS rn
    FROM totals
)
SELECT category, product, total_quantity, rn
FROM ranked
WHERE rn <= 2
ORDER BY category, rn;
'''

print("Dedup:", cur.execute(dedup_sql).fetchall())
print("Latest:", cur.execute(latest_sql).fetchall())
print("No events:", cur.execute(anti_join_sql).fetchall())
print("Pivot:", cur.execute(pivot_sql).fetchall())
print("Top N:", cur.execute(top_n_sql).fetchall())

assert cur.execute(anti_join_sql).fetchall() == [('u4',)]
conn.close()

---
# Tonight’s Priority Order

Do these first:

1. Two Sum
2. Binary Search
3. Subarray Sum Equals K
4. Sliding Window Maximum
5. Merge/Insert Intervals
6. Course Schedule II
7. Graph Valid Tree
8. Koko / Ship Capacity
9. Coin Change
10. Watermark + Late Events
11. Streaming Median
12. Sessionization
13. SQL dedup/latest/anti-join/top-N
14. LRU + Backpressure
15. Bloom Filter

## Final memory line

**Clarify → simple solution → bottleneck → optimized pattern → invariant → example → complexity → edge cases**